# Phase 2: Feature Engineering

**Goal:** Impute missing values, create derived features, split into train/test, and save the processed dataset.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_raw, save_processed
from src.features.engineer import engineer, split, TARGET, FEATURES

sns.set_theme(style='whitegrid', palette='muted')

## 1. Missing Values — Before

In [ ]:
df_raw = load_raw()
print(f"Shape: {df_raw.shape}")

missing = df_raw.isnull().sum()
missing[missing > 0]

## 2. Apply Feature Engineering

In [ ]:
df = engineer(df_raw)

print("Missing values after imputation:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else "None")

## 3. Derived Features

In [ ]:
engineered_cols = ['debt_to_income', 'total_past_due', 'income_per_dependent']
df[engineered_cols].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(engineered_cols):
    upper = df[col].quantile(0.99)
    df[col].clip(upper=upper).hist(ax=axes[i], bins=40,
                                   color='#55A868', edgecolor='white')
    axes[i].set_title(col, fontsize=10)

plt.suptitle('Engineered Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Default Rate — Engineered Features

Validate that derived features carry predictive signal before committing to them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(engineered_cols):
    clipped = df[col].clip(upper=df[col].quantile(0.99))
    bins = pd.qcut(clipped, q=10, duplicates='drop')
    rate = df.groupby(bins, observed=True)[TARGET].mean() * 100
    rate.plot(kind='bar', ax=axes[i], color='#DD8452', edgecolor='white')
    axes[i].set_title(f'Default Rate by {col}', fontsize=9)
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('Default Rate by Engineered Feature Decile', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = split(df)

print(f"Train : {X_train.shape[0]:>6,} rows  |  default rate: {y_train.mean():.4f}")
print(f"Test  : {X_test.shape[0]:>6,} rows  |  default rate: {y_test.mean():.4f}")
print(f"\nFeatures used ({len(FEATURES)}):")
for f in FEATURES:
    print(f"  {f}")

## 6. Save Processed Dataset

In [ ]:
save_processed(df)
print("Saved → data/processed/processed.parquet")